# 07 · Ergebnisse: Selection Bias und PriDe

Dieses Notebook liest **vorhandene Summaries**. Es startet keine Modellläufe,
lädt keine Modelle herunter und reicht keine Jobs ein. Standardmäßig werden
nur echte, nicht limitierte Analysen ausgewählt. Ohne lokale LRZ-Ergebnisse
bleiben die Tabellen und Plots leer; es werden keine Ergebnisse erfunden.

Benötigte Pakete sind in `pip install -e '.[notebooks]'` enthalten.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for base in candidates for p in (base, base / "analysis")
                     if (p / "src").is_dir() and (p / "pyproject.toml").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Notebook im MCQ-Projekt öffnen.")
for path in (PROJECT_ROOT, PROJECT_ROOT / "scripts"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print("Projekt:", PROJECT_ROOT)


Projekt: /Users/diergexiaohai/Desktop/MCQ/analysis


## 1 · Ergebnisse auswählen

LRZ-Ergebnisse müssen auf dem Rechner liegen, auf dem dieses Notebook läuft.
Die gleiche Ordnerstruktur unter `outputs/summaries` kann vom LRZ kopiert werden.
Alternativ `SUMMARY_DIR` anpassen. Für diesen Bericht reichen die Summary-CSVs;
für die Prüfung einzelner Antworten zusätzlich Rohdaten und Metadaten kopieren.

Zunächst zeigt die Inventartabelle alle verfügbaren verwalteten Pipelines.
Danach in `PIPELINE_IDS` genau eine Pipeline pro Modell × Datensatz eintragen.
Keine automatische Auswahl des neuesten Laufs: So werden Wiederholungen,
Mock-Tests und volle Analysen nicht unbemerkt vermischt.


In [2]:
SUMMARY_DIR = PROJECT_ROOT / "outputs" / "summaries"
PIPELINE_IDS = []  # IDs aus der Inventartabelle unten hier eintragen.
SCORER = "real"  # Nur zur technischen Demonstration explizit auf "mock" ändern.
ALLOW_LIMITED = False  # Für einen kleinen Test bewusst auf True setzen.
EXPORT = False  # Bei True: neue CSV sowie PNG/PDF im eigenen Berichtsordner.


In [3]:
required = {"pipeline_id", "kind", "scorer_backend", "run_dir"}
frames = []
skipped = []
for path in sorted(SUMMARY_DIR.glob("*.csv"), key=lambda p: p.stat().st_mtime_ns):
    try:
        frame = pd.read_csv(path)
    except (pd.errors.EmptyDataError, pd.errors.ParserError) as exc:
        skipped.append((path.name, str(exc)))
        continue
    if required.issubset(frame.columns):
        frames.append(frame)

summaries = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if not summaries.empty:
    summaries = summaries.drop_duplicates("run_dir", keep="last")
    managed = summaries[summaries.pipeline_id.notna() & (summaries.pipeline_id != "unmanaged")]
    inventory = managed.groupby(["pipeline_id", "model_repo", "dataset_repo", "scorer_backend"],
                                dropna=False).agg(
        Schritte=("kind", lambda values: ", ".join(sorted(set(values)))),
        Summary_Zeilen=("kind", "size"),
        Limit=("selection_limit", lambda values: ", ".join(sorted({
            "voll" if pd.isna(x) else str(int(x)) for x in values}))),
    ).reset_index()
    display(inventory)
else:
    print("Noch keine passenden Summary-Dateien vorhanden.")
if skipped:
    print("Nicht lesbare Dateien:", skipped)


,pipeline_id,model_repo,dataset_repo,scorer_backend,Schritte,Summary_Zeilen,Limit
0,mock_qwen3-8b_medmcqa_39b71df624e1,Qwen/Qwen3-8B,openlifescienceai/medmcqa,mock,"baseline, permutations, pride",3,8
1,mock_qwen3-8b_medqa_7110cce5f874,Qwen/Qwen3-8B,openlifescienceai/MedQA-USMLE-4-options-hf,mock,"baseline, permutations, pride",3,8
2,mock_qwen3-8b_medqa_c2b288669e93,Qwen/Qwen3-8B,openlifescienceai/MedQA-USMLE-4-options-hf,mock,"baseline, permutations, pride",3,8
3,mock_qwen3-8b_mmlu_e25027fc5e7d,Qwen/Qwen3-8B,cais/mmlu,mock,"baseline, permutations, pride",3,8


## 2 · Zusammengehörige Ergebnisse laden

Die bestehende Aggregation prüft Pipeline-ID, Backend, Modell, Datensatz, Seed
und Stichproben-Fingerprint. Baseline, Permutationen und PriDe müssen zusammen
vorliegen. Bei doppelten Modell-Datensatz-Paaren oder unerlaubten kleinen Läufen
bricht diese Zelle ab, statt einen mehrdeutigen Vergleich zu zeigen.

Die Stichprobengröße bezieht sich auf Fragen. Bei Permutationen enthält `n`
hingegen vier Vorhersagen pro Frage. Alle Accuracy- und RStd-Werte werden intern
auf der Skala 0–1 geführt; Accuracy-Deltas werden unten in Prozentpunkten gezeigt.


In [4]:
from aggregate_results import aggregate

comparison = pd.DataFrame()
if SCORER not in {"real", "mock"}:
    raise ValueError("SCORER muss real oder mock sein.")
if len(PIPELINE_IDS) != len(set(PIPELINE_IDS)):
    raise ValueError("Jede Pipeline-ID nur einmal auswählen.")
if PIPELINE_IDS:
    comparison = pd.concat([aggregate(SUMMARY_DIR, pid, SCORER) for pid in PIPELINE_IDS],
                           ignore_index=True)
    if comparison.duplicated(["model_repo", "dataset_repo"]).any():
        raise ValueError("Bitte genau eine Pipeline pro Modell × Datensatz auswählen.")
    limits = ["baseline_selection_limit", "permutations_selection_limit", "pride_selection_limit"]
    if not set(limits).issubset(comparison.columns):
        raise ValueError("Limit-Metadaten fehlen: Ergebnisse zunächst prüfen.")
    if not ALLOW_LIMITED and comparison[limits].notna().any().any():
        raise ValueError("Limitierte Läufe ausgewählt. Für Testansicht ALLOW_LIMITED=True setzen.")
    if not (comparison["baseline_n"] == comparison["pride_pride_n"]).all():
        raise ValueError("Baseline und PriDe haben unterschiedliche Fragezahlen.")
    if not (comparison["permutations_n"] == 4 * comparison["baseline_n"]).all():
        raise ValueError("Es fehlen Permutationsvorhersagen.")
    display(comparison[["pipeline_id", "model_repo", "dataset_repo", "seed",
                        "sample_set_sha256", "baseline_n"]])
else:
    print("Noch keine Auswahl: PIPELINE_IDS oben ausfüllen und ab dieser Zelle neu ausführen.")

report_label = "MOCK – keine Modellresultate" if SCORER == "mock" else (
    "TECHNISCHER TEST – limitierte Läufe zugelassen" if ALLOW_LIMITED else "Echte vollständige Analysen")
print(report_label)


Noch keine Auswahl: PIPELINE_IDS oben ausfüllen und ab dieser Zelle neu ausführen.
Echte vollständige Analysen


## 3 · Vergleichstabelle

**ΔAccuracy > 0** bedeutet höhere Trefferquote nach PriDe.
**ΔRStd < 0** bedeutet geringere Streuung der vier positionsbezogenen Recalls.
Eine Verringerung von RStd allein garantiert keine bessere Accuracy.

Die PriDe-Gesamtmetrik umfasst die zyklisch ausgewerteten Fragen aus D_e und
die prior-korrigierten Fragen aus D_r. Die Tabelle weist keine separaten
D_e/D_r-Metriken oder statistischen Konfidenzintervalle aus.
`NaN` bleibt sichtbar: Ohne alle vier Goldklassen ist RStd nicht bestimmbar.


In [5]:
table = pd.DataFrame()
if not comparison.empty:
    table = pd.DataFrame({
        "Modell": comparison.model_repo,
        "Datensatz": comparison.dataset_repo,
        "Fragen": comparison.baseline_n.astype(int),
        "Accuracy vorher (%)": 100 * comparison.baseline_accuracy,
        "Accuracy PriDe (%)": 100 * comparison.pride_pride_accuracy,
        "ΔAccuracy (pp)": 100 * comparison.pride_delta_accuracy,
        "RStd vorher": comparison.baseline_rstd,
        "RStd PriDe": comparison.pride_pride_rstd,
        "ΔRStd": comparison.pride_delta_rstd,
        "Permutation-Spread (pp)": 100 * comparison.permutations_accuracy_permutation_range,
    })
    display(table.round(4))
else:
    print("Keine ausgewählten Ergebnisse.")


Keine ausgewählten Ergebnisse.


## 4 · Plots

Die Abbildungen zeigen Accuracy, RStd, A–D-Antwortanteile und Accuracy je
Permutation. Es werden keine Fehlerbalken aus einem einzelnen Lauf abgeleitet.
Die Antwortanteile werden als Beschreibung gezeigt, nicht als Beweis dafür,
dass 25 % pro Option für jeden Datensatz die richtige Zielverteilung wäre.


In [6]:
import matplotlib.pyplot as plt

figures = []
for _, row in comparison.iterrows():
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), layout="constrained")
    names = ["Baseline", "PriDe"]
    axes[0, 0].bar(names, 100 * np.array([row.baseline_accuracy, row.pride_pride_accuracy]))
    axes[0, 0].set(ylabel="Accuracy (%)", ylim=(0, 100))
    rstd = np.array([row.baseline_rstd, row.pride_pride_rstd], dtype=float)
    axes[0, 1].bar(names, rstd)
    axes[0, 1].set(ylabel="RStd (0–1)", ylim=(0, max(0.05, float(np.nanmax(rstd)) * 1.2)
                                              if np.isfinite(rstd).any() else 0.05))
    if not np.isfinite(rstd).all():
        axes[0, 1].text(0.5, 0.8, "RStd teilweise/nicht bestimmbar",
                        transform=axes[0, 1].transAxes, ha="center")
    x = np.arange(4)
    for offset, prefix, label in [(-0.18, "baseline", "Baseline"),
                                  (0.18, "pride_pride", "PriDe")]:
        values = [100 * row[f"{prefix}_prediction_frequency_{c}"] for c in "ABCD"]
        axes[1, 0].bar(x + offset, values, width=0.36, label=label)
    axes[1, 0].set(xticks=x, xticklabels=list("ABCD"), ylabel="Antwortanteil (%)", ylim=(0, 100))
    axes[1, 0].legend()
    axes[1, 1].plot(x, [100 * row[f"permutations_accuracy_permutation_{i}"] for i in x], marker="o")
    axes[1, 1].set(xticks=x, xlabel="Zyklische Permutation", ylabel="Accuracy (%)", ylim=(0, 100))
    fig.suptitle(f"{report_label}\n{row.model_repo} · {row.dataset_repo}", fontsize=10)
    figures.append((row.pipeline_id, fig))
    plt.show()
if comparison.empty:
    print("Plots erscheinen nach Auswahl gespeicherter Ergebnisse.")


Matplotlib is building the font cache; this may take a moment.


Plots erscheinen nach Auswahl gespeicherter Ergebnisse.


## 5 · Optional exportieren

Bei `EXPORT=True` wird ein neuer Berichtsordner erzeugt. Die vollständige Tabelle
behält Pipeline-ID, Fingerprint und Metadaten; zusätzlich werden die kompakte
Tabelle und Abbildungen als PNG und PDF gespeichert. Ursprüngliche Ergebnisse
werden nicht überschrieben. Mock- und Testberichte sind separat benannt.


In [7]:
if EXPORT and not comparison.empty:
    from datetime import datetime, timezone
    import uuid
    mode = "mock" if SCORER == "mock" else ("test" if ALLOW_LIMITED else "real-full")
    report_dir = PROJECT_ROOT / "outputs" / "reports" / (
        mode + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
    report_dir.mkdir(parents=True, exist_ok=False)
    comparison.to_csv(report_dir / "comparison_with_metadata.csv", index=False)
    table.to_csv(report_dir / "comparison_readable.csv", index=False)
    for index, (pipeline_id, fig) in enumerate(figures):
        for extension in ("png", "pdf"):
            fig.savefig(report_dir / f"comparison_{index + 1}.{extension}", dpi=180, bbox_inches="tight")
    print("Export:", report_dir)
else:
    print("Kein Export angefordert oder keine Ergebnisse ausgewählt.")


Kein Export angefordert oder keine Ergebnisse ausgewählt.
